# LangGraph + LangChain Stateful Execution Architecture

**Refactoring the hanchan multi-agent system into a graph-based execution engine**

This notebook is the design document for restructuring `src/fastapi/` from the current
orchestrator-worker model into a clean LangGraph state graph with LangChain-powered subagents.

**Design constraints:**
- No hardcoded data models, schemas, or database structures
- Subagents are reusable outside LangGraph and do not depend on graph structure
- Memory patterns follow `08_episodic_with_semantic.ipynb`
- LangChain doc patterns (Qdrant, HITL, SQL, Neo4j, Guardrails) are reflected
- System remains modular and every component is replaceable

**Current state (before):** `src/fastapi/app/agents/memory_agent/` uses a LangGraph graph
with orchestrator → worker → tools → reviewer → generator flow, custom routing, and
tightly-coupled node implementations.

**Target state (after):** Clean separation into graph topology, thin adapter nodes,
standalone subagents, and service abstractions.

---
# 1. Documentation Analysis & Pattern Extraction

Before designing, we analyze the reference docs shipped with the repo:
- `docs/langchain-docs/` — Qdrant, HITL, SQL, Neo4j, Guardrails
- `docs/notebooks/08_episodic_with_semantic.ipynb` — episodic + semantic memory

In [ ]:
"""
Phase 1: Parse documentation files and extract key patterns.
"""
from pathlib import Path

DOCS = Path("/home/tra01/project/hanchan/docs/langchain-docs")
doc_files = sorted(DOCS.iterdir())

patterns: dict[str, list[str]] = {}
for f in doc_files:
    text = f.read_text()
    headers = [line.strip().lstrip("#").strip() for line in text.splitlines() if line.strip().startswith("#")]
    patterns[f.name] = headers

print("=== Documentation Pattern Map ===\n")
for doc, heads in patterns.items():
    print(f"📄 {doc}")
    for h in heads[:8]:
        print(f"   • {h}")
    if len(heads) > 8:
        print(f"   ... +{len(heads)-8} more")
    print()

In [ ]:
"""
Summarize critical integration patterns from all docs.
These drive every design decision in the architecture.
"""

DOC_PATTERNS = {
    "qdrant": {
        "initialization": "QdrantVectorStore with client, collection_name, embedding",
        "cloud_connect": "QdrantVectorStore.from_existing_collection(url=, api_key=, prefer_grpc=True)",
        "retrieval": "similarity_search(query, k, filter) or .as_retriever(search_kwargs={'k': N})",
        "add_docs": "vector_store.add_documents(documents, ids) — UUIDs recommended",
        "abstraction": "Behind LangChain Retriever interface — swappable",
        "key_tip": "prefer_grpc=True for remote; use metadata filtering per user_id",
    },
    "human_in_the_loop": {
        "mechanism": "LangGraph interrupt() inside nodes — most flexible",
        "static_breakpoints": "compile(interrupt_before=['node_name'])",
        "dynamic_interrupt": "interrupt(payload) inside node — conditional pause",
        "resume": "graph.invoke(Command(resume=data), config)",
        "requirement": "Persistent checkpointer (MemorySaver, PostgresSaver, etc.)",
        "best_practice": "Confidence-based escalation — interrupt only when uncertain",
    },
    "sql": {
        "toolkit": "SQLDatabaseToolkit(db, llm) → 4 tools: list, schema, query, checker",
        "agent": "create_tool_calling_agent(llm, tools, prompt) with AgentExecutor",
        "db_connect": "SQLDatabase.from_uri(connection_string) — any SQLAlchemy dialect",
        "safety": "Read-only credentials, minimal schema exposure, query timeout",
        "retry": "Agent self-corrects SQL errors via QuerySQLCheckerTool",
    },
    "neo4j": {
        "connect": "Neo4jGraph(url, username, password)",
        "cypher_qa": "GraphCypherQAChain.from_llm(llm, graph) — NL → Cypher → answer",
        "kg_build": "LLMGraphTransformer for text → knowledge graph",
        "safety": "Low-privilege DB user for LLM-generated Cypher",
    },
    "guardrails": {
        "pii": "PIIMiddleware(pii_type, strategy=[redact|mask|hash|block])",
        "content_filter": "Custom AgentMiddleware with before_agent / after_agent hooks",
        "layered": "Cheap deterministic first → model-based last",
        "configurable": "Strategies and detectors passed as config, not hardcoded",
    },
    "memory_notebook": {
        "episodic": "Vector store of interaction summaries — 'what happened'",
        "semantic": "Neo4j graph of entities/relationships — 'what is known'",
        "retrieval": "Dual-source: similarity_search + fulltext cypher, merged into context",
        "update": "Post-response: summarize interaction → episodic; extract KG → semantic",
        "lifecycle": "retrieve → generate → update (never update mid-reasoning)",
    },
}

for category, items in DOC_PATTERNS.items():
    print(f"\n{'='*60}")
    print(f"  {category.upper()}")
    print(f"{'='*60}")
    for k, v in items.items():
        print(f"  {k:20s} │ {v}")

---
# 2. Shared Execution State Schema

The state is a `TypedDict` shared across all nodes. It must be:
- **Extensible** — new fields don't break existing nodes
- **Dedup-aware** — tracks executed operations
- **Memory-integrated** — episodic + semantic refs from notebook patterns

In [ ]:
"""
Shared Execution State — the single source of truth flowing through the graph.

Design principles:
  1. Annotated list fields use reducer (append-merge) so nodes return partials
  2. Set-based `executed_ops` prevents duplicate tool / query execution
  3. Memory references are typed to distinguish episodic vs semantic sources
  4. `extras` dict provides escape hatch for future extensions
"""
from __future__ import annotations
from typing import Annotated, Any
from langchain_core.messages import BaseMessage
from typing_extensions import TypedDict


# --- Reducers ---
def _merge_lists(existing: list, new: list) -> list:
    """Append-merge reducer for list fields."""
    return existing + new

def _merge_sets(existing: set, new: set) -> set:
    """Union-merge reducer for set fields."""
    return existing | new

def _merge_dicts(existing: dict, new: dict) -> dict:
    """Shallow-merge reducer for dict fields."""
    return {**existing, **new}


class GraphState(TypedDict, total=False):
    """Shared execution state for the LangGraph pipeline.
    Nodes read what they need and return partial updates.
    Fields with Annotated reducers auto-merge on update.
    """
    # ── Identity ──────────────────────────────────────────
    user_id: str
    session_id: str | None
    jwt: str

    # ── Input ─────────────────────────────────────────────
    user_input: str                          # original (pre-guard) user message
    sanitized_input: str                     # post-PII-masking input
    messages: Annotated[list[BaseMessage], _merge_lists]

    # ── Routing & Control ─────────────────────────────────
    route_decision: str                      # "memory" | "fsrs" | "sql" | "direct"
    current_step: str
    step_count: int                          # total node invocations (for max-step guard)
    router_loops: int                        # times we've looped back to router
    iterations: int

    # ── Intermediate Results ──────────────────────────────
    intermediate_results: Annotated[dict[str, Any], _merge_dicts]
    executed_ops: Annotated[set[str], _merge_sets]  # dedup: "sql:SELECT ...", "mem:query_xyz"

    # ── Memory Context (notebook pattern) ─────────────────
    retrieved_episodic: list[dict[str, Any]]     # [{content, score, created_at}, ...]
    retrieved_semantic: list[dict[str, Any]]     # [{subject, predicate, object, score}, ...]
    memory_context: str                          # merged text block for LLM prompt

    # ── SQL ───────────────────────────────────────────────
    sql_results: list[dict[str, Any]]

    # ── Guardrails ────────────────────────────────────────
    input_guard_passed: bool
    output_guard_passed: bool
    guard_violations: Annotated[list[str], _merge_lists]

    # ── Human-in-the-Loop ─────────────────────────────────
    needs_human_review: bool
    human_feedback: str | None

    # ── Output ────────────────────────────────────────────
    generation: str
    audio_file: str | None
    tts_enabled: bool

    # ── Timing / Observability ────────────────────────────
    start_time: float
    metadata: Annotated[dict[str, Any], _merge_dicts]

    # ── Escape Hatch ──────────────────────────────────────
    extras: Annotated[dict[str, Any], _merge_dicts]


# --- Example: partial state update from a node ---
example_partial = {
    "route_decision": "memory",
    "step_count": 2,
    "intermediate_results": {"router_confidence": 0.92},
}
print("GraphState defined with", len(GraphState.__annotations__), "fields")
print("Example partial update:", example_partial)

---
# 3. Subagent Interface & Registry

Subagents are **standalone LangChain components** that:
- Accept a plain `dict` context, return a plain `dict` result
- Wrap LangChain primitives (chains, tools, retrievers) internally
- Are fully testable and usable **outside** the LangGraph graph
- Are registered in a factory so they can be swapped at runtime

In [ ]:
"""
Subagent protocol + factory registry.

Every subagent implements: invoke(context: dict) -> dict
Nodes call them uniformly; tests mock them trivially.
"""
from __future__ import annotations
from typing import Any, Protocol, runtime_checkable


@runtime_checkable
class Subagent(Protocol):
    """Contract every subagent must satisfy."""
    name: str

    def invoke(self, context: dict[str, Any]) -> dict[str, Any]:
        """Execute the subagent's capability.
        Args: context — flat dict with relevant inputs
        Returns: flat dict with results
        """
        ...


class SubagentRegistry:
    """Factory/registry for subagents. Enables runtime swapping."""

    def __init__(self) -> None:
        self._agents: dict[str, Subagent] = {}

    def register(self, agent: Subagent) -> None:
        self._agents[agent.name] = agent

    def get(self, name: str) -> Subagent:
        if name not in self._agents:
            raise KeyError(f"Subagent '{name}' not registered. Available: {list(self._agents)}")
        return self._agents[name]

    def replace(self, name: str, agent: Subagent) -> None:
        """Hot-swap a subagent (e.g. for testing or A/B experiments)."""
        self._agents[name] = agent

    def list_agents(self) -> list[str]:
        return list(self._agents)


# Global registry — populated at app startup
registry = SubagentRegistry()

print("Subagent protocol defined. Registry ready.")
print("Interface: invoke(context: dict) -> dict")

---
# 4. Guardrail & PII Subagents

From the guardrails doc: **deterministic first** (regex, blocklist) → **model-based second** (LLM judge).
Both configurable via external config, not hardcoded rules.

In [ ]:
"""
Guardrail subagent — configurable input/output safety validation.
PII subagent — configurable PII detection and masking.
Both standalone: no dependency on LangGraph state or graph structure.
"""
from __future__ import annotations
import re
from dataclasses import dataclass, field
from typing import Any


# ── Guardrail Subagent ────────────────────────────────────

@dataclass
class GuardrailConfig:
    """Externally-provided config — nothing hardcoded."""
    blocked_patterns: list[str] = field(default_factory=list)
    blocked_keywords: list[str] = field(default_factory=list)
    max_input_length: int = 10_000
    enable_llm_judge: bool = False


class GuardrailSubagent:
    name = "guardrail"

    def __init__(self, config: GuardrailConfig, llm=None):
        self.config = config
        self._compiled = [re.compile(p, re.IGNORECASE) for p in config.blocked_patterns]
        self._llm = llm

    def invoke(self, context: dict[str, Any]) -> dict[str, Any]:
        text = context.get("text", "")
        violations: list[str] = []

        # Layer 1: Length check
        if len(text) > self.config.max_input_length:
            violations.append(f"input_too_long:{len(text)}")

        # Layer 2: Blocked keywords (cheap)
        text_lower = text.lower()
        for kw in self.config.blocked_keywords:
            if kw.lower() in text_lower:
                violations.append(f"blocked_keyword:{kw}")

        # Layer 3: Regex patterns
        for pattern in self._compiled:
            if pattern.search(text):
                violations.append(f"blocked_pattern:{pattern.pattern}")

        # Layer 4: LLM judge (expensive — only if enabled and no violations yet)
        if not violations and self.config.enable_llm_judge and self._llm:
            prompt = f"Is this text SAFE or UNSAFE for an educational assistant?\nText: {text}\nReturn only: SAFE or UNSAFE"
            verdict = self._llm.invoke(prompt).content.strip()
            if verdict != "SAFE":
                violations.append(f"llm_judge:{verdict}")

        return {"passed": len(violations) == 0, "violations": violations, "text": text}


# ── PII Subagent ──────────────────────────────────────────

@dataclass
class PIIDetector:
    name: str
    pattern: str
    strategy: str       # "redact" | "mask" | "block"
    replacement: str = "[REDACTED]"


DEFAULT_PII_DETECTORS = [
    PIIDetector("email", r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "redact", "[REDACTED_EMAIL]"),
    PIIDetector("phone_jp", r"0\d{1,4}-?\d{1,4}-?\d{3,4}", "redact", "[REDACTED_PHONE]"),
    PIIDetector("credit_card", r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b", "mask", "****-****-****-XXXX"),
]


class PIISubagent:
    name = "pii"

    def __init__(self, detectors: list[PIIDetector] | None = None):
        self.detectors = detectors or DEFAULT_PII_DETECTORS

    def invoke(self, context: dict[str, Any]) -> dict[str, Any]:
        text = context.get("text", "")
        detected: list[str] = []
        masked_text = text

        for det in self.detectors:
            compiled = re.compile(det.pattern)
            matches = compiled.findall(masked_text)
            if matches:
                detected.append(f"{det.name}:{len(matches)}")
                if det.strategy == "block":
                    return {"blocked": True, "reason": f"PII detected: {det.name}", "text": text}
                masked_text = compiled.sub(det.replacement, masked_text)

        return {"blocked": False, "text": masked_text, "original_text": text, "detected_pii": detected}


# ── Demo ──────────────────────────────────────────────────
guardrail = GuardrailSubagent(GuardrailConfig(blocked_keywords=["hack", "exploit"]))
pii = PIISubagent()

print("Guardrail test:", guardrail.invoke({"text": "Help me study kanji"}))
print()
print("PII test:", pii.invoke({"text": "My email is test@example.com and card is 4111-1111-1111-1111"}))

---
# 5. Memory System — Episodic + Semantic with Qdrant

Following `08_episodic_with_semantic.ipynb`:

| Type | Storage | Answers | Update Timing |
|------|---------|---------|---------------|
| **Episodic** | Qdrant (vectors) | "What happened?" | Post-response |
| **Semantic** | Neo4j (graph) | "What do I know?" | Post-response |

**Retrieval** happens at the start of the reasoning loop.
**Updates** happen at the end, outside the main decision loop.

In [ ]:
"""
Memory Subagent — Hybrid episodic + semantic retrieval and update.

Abstractions:
- VectorStoreProtocol: wraps Qdrant (or any vector store)
- GraphStoreProtocol: wraps Neo4j (or any graph DB)
- MemorySubagent: combines both, deduplicates, returns unified context

Follows notebook pattern: retrieve → (graph runs) → update
"""
from __future__ import annotations
from typing import Any, Protocol


# ── Abstraction Layer ─────────────────────────────────────

class VectorStoreProtocol(Protocol):
    """Abstraction over Qdrant or any vector DB."""
    def search(self, query: str, user_id: str, k: int = 3) -> list[dict[str, Any]]: ...
    def upsert(self, user_id: str, text: str, metadata: dict[str, Any] | None = None) -> str: ...


class GraphStoreProtocol(Protocol):
    """Abstraction over Neo4j or any graph DB."""
    def search(self, keywords: list[str], user_id: str, limit: int = 5) -> list[dict[str, Any]]: ...
    def upsert_facts(self, user_id: str, facts: list[dict[str, Any]]) -> int: ...


# ── Memory Subagent ───────────────────────────────────────

class MemorySubagent:
    """Combines episodic (vector) + semantic (graph) memory.
    Retrieval: merges both sources, deduplicates via already_seen IDs.
    Update: writes new episodic summary + extracted semantic facts.
    """
    name = "memory"

    def __init__(self, vector_store: VectorStoreProtocol, graph_store: GraphStoreProtocol, llm=None):
        self._vector = vector_store
        self._graph = graph_store
        self._llm = llm

    def invoke(self, context: dict[str, Any]) -> dict[str, Any]:
        """Retrieve memories for the current query."""
        query = context["query"]
        user_id = context["user_id"]
        already_seen = context.get("already_seen", set())
        k = context.get("k", 3)

        # 1. Episodic retrieval (Qdrant similarity search)
        episodic_raw = self._vector.search(query, user_id, k=k)
        episodic = [e for e in episodic_raw if e.get("id") not in already_seen]

        # 2. Semantic retrieval (Neo4j keyword/entity match)
        keywords = query.split()[:5]
        semantic_raw = self._graph.search(keywords, user_id)
        semantic = [s for s in semantic_raw if s.get("id") not in already_seen]

        # 3. Merge into unified context string
        ep_text = "\n".join(f"- {e.get('content', '')}" for e in episodic) or "(no episodic memories)"
        sem_text = "\n".join(
            f"- ({s.get('subject','?')}) -[{s.get('predicate','?')}]-> ({s.get('object','?')})"
            for s in semantic
        ) or "(no semantic facts)"

        memory_context = (
            f"**Past Interactions (Episodic):**\n{ep_text}\n\n"
            f"**Known Facts (Semantic):**\n{sem_text}"
        )

        new_ids = {e.get("id") for e in episodic} | {s.get("id") for s in semantic}

        return {
            "retrieved_episodic": episodic,
            "retrieved_semantic": semantic,
            "memory_context": memory_context,
            "new_memory_ids": new_ids,
        }

    def update(self, context: dict[str, Any]) -> dict[str, Any]:
        """Post-response memory update (called from post_update_node)."""
        user_id = context["user_id"]
        user_input = context["user_input"]
        generation = context["generation"]
        results = {"episodic_written": False, "semantic_facts_written": 0}

        # Episodic: summarize and store
        if self._llm:
            interaction = f"User: {user_input}\nAssistant: {generation}"
            summary = self._llm.invoke(f"Summarize this interaction in one sentence:\n{interaction}").content
        else:
            summary = f"User asked: {user_input[:100]}..."

        doc_id = self._vector.upsert(user_id, summary, {"type": "episodic"})
        results["episodic_written"] = True
        results["episodic_doc_id"] = doc_id

        # Semantic: extract facts and write to graph (production: LLM structured output)
        return results


print("MemorySubagent defined with:")
print("  .invoke(context)  → retrieval (episodic + semantic)")
print("  .update(context)  → post-response write")
print()
print("Backed by:")
print("  VectorStoreProtocol → Qdrant (existing: services/memory/episodic_memory.py)")
print("  GraphStoreProtocol  → Neo4j  (existing: services/memory/semantic_memory.py)")

## 6. SQL Subagent

Uses `SQLDatabaseToolkit` from LangChain with a tool-calling agent.  
4 built-in tools: `sql_db_list_tables`, `sql_db_schema`, `sql_db_query`, `sql_db_query_checker`.  
Wraps in retry loop with deduplication to prevent re-running identical queries.

In [ ]:
"""
SQL Subagent — Structured database access via LangChain SQL toolkit.

Pattern from docs: SQLDatabaseToolkit → create_tool_calling_agent → AgentExecutor
We add: retry loop, query dedup, and result normalization.
"""
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_tool_calling_agent, AgentExecutor


class SQLSubagent:
    """Wraps LangChain SQL toolkit with retry + dedup."""
    name = "sql"

    SQL_SYSTEM_PROMPT = (
        "You are a database assistant. Use the provided SQL tools to answer questions.\n"
        "NEVER modify data (no INSERT, UPDATE, DELETE). Only SELECT queries.\n"
        "Always verify the schema before querying. Use sql_db_query_checker before executing."
    )

    def __init__(self, db_uri: str, llm, max_retries: int = 2, include_tables: list[str] | None = None):
        db_kwargs = {}
        if include_tables:
            db_kwargs["include_tables"] = include_tables
        self._db = SQLDatabase.from_uri(db_uri, **db_kwargs)
        self._llm = llm
        self._max_retries = max_retries

        # LangChain SQL toolkit gives us 4 tools
        toolkit = SQLDatabaseToolkit(db=self._db, llm=llm)
        tools = toolkit.get_tools()

        prompt = ChatPromptTemplate.from_messages([
            ("system", self.SQL_SYSTEM_PROMPT),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}"),
        ])
        agent = create_tool_calling_agent(llm, tools, prompt)
        self._executor = AgentExecutor(agent=agent, tools=tools, verbose=False)
        self._query_cache: dict[str, str] = {}

    def invoke(self, context: dict[str, Any]) -> dict[str, Any]:
        """Execute SQL retrieval with retry and dedup."""
        query = context["query"]
        user_id = context["user_id"]

        # Dedup: skip if exact query already executed this session
        cache_key = f"{user_id}:{query}"
        if cache_key in self._query_cache:
            return {"retrieved_sql": self._query_cache[cache_key], "sql_cached": True}

        last_error = None
        for attempt in range(self._max_retries + 1):
            try:
                result = self._executor.invoke({"input": query})
                output = result.get("output", "")
                self._query_cache[cache_key] = output
                return {"retrieved_sql": output, "sql_cached": False}
            except Exception as exc:
                last_error = exc
                continue

        return {"retrieved_sql": f"SQL error after {self._max_retries+1} attempts: {last_error}", "sql_error": True}


print("SQLSubagent defined:")
print("  - 4 built-in tools from SQLDatabaseToolkit")
print("  - Read-only enforcement via system prompt")
print("  - Query dedup + retry loop")
print("  - Maps to existing: services/sql/ and nodes/workers.py sql_worker")

## 7. Neo4j Subagent

Uses `Neo4jGraph` + `GraphCypherQAChain` for natural language → Cypher → structured facts.  
Also uses `LLMGraphTransformer` on the write path to extract entities + relationships from conversation text.

In [ ]:
"""
Neo4j Subagent — Graph-based semantic knowledge via Cypher QA chain.

Read path:  natural language → Cypher query → structured results (GraphCypherQAChain)
Write path: conversation text → LLMGraphTransformer → graph_documents → add_graph_documents
"""
from langchain_community.graphs import Neo4jGraph
from langchain.chains import GraphCypherQAChain
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document


class Neo4jSubagent:
    """Wraps Neo4j for semantic knowledge retrieval and fact extraction."""
    name = "neo4j"

    CYPHER_SYSTEM = (
        "You generate Cypher queries for a Neo4j knowledge graph.\n"
        "The graph stores facts about a language-learning user: "
        "known vocabulary, grammar patterns, learning progress, and personal facts.\n"
        "Always filter by user_id property when present."
    )

    def __init__(self, uri: str, user: str, password: str, llm):
        self._graph = Neo4jGraph(url=uri, username=user, password=password)
        self._llm = llm
        self._qa_chain = GraphCypherQAChain.from_llm(
            llm=llm,
            graph=self._graph,
            verbose=False,
            allow_dangerous_requests=True,  # required for Cypher generation
        )
        self._transformer = LLMGraphTransformer(llm=llm)

    def invoke(self, context: dict[str, Any]) -> dict[str, Any]:
        """Read path: natural language → Cypher → answer."""
        query = context["query"]
        user_id = context["user_id"]

        try:
            scoped_query = f"For user {user_id}: {query}"
            result = self._qa_chain.invoke({"query": scoped_query})
            return {
                "retrieved_semantic": result.get("result", ""),
                "cypher_used": result.get("intermediate_steps", [{}])[0].get("query", ""),
            }
        except Exception as exc:
            return {"retrieved_semantic": f"Neo4j error: {exc}", "neo4j_error": True}

    def update(self, context: dict[str, Any]) -> dict[str, Any]:
        """Write path: extract facts from conversation and add to graph."""
        user_input = context["user_input"]
        generation = context["generation"]
        user_id = context["user_id"]

        interaction_text = f"User: {user_input}\nAssistant: {generation}"
        doc = Document(page_content=interaction_text, metadata={"user_id": user_id})

        try:
            graph_docs = self._transformer.convert_to_graph_documents([doc])
            if graph_docs:
                self._graph.add_graph_documents(graph_docs)
            return {"semantic_facts_written": sum(len(gd.relationships) for gd in graph_docs)}
        except Exception as exc:
            return {"semantic_facts_written": 0, "neo4j_update_error": str(exc)}


print("Neo4jSubagent defined:")
print("  Read:  GraphCypherQAChain (NL → Cypher → results)")
print("  Write: LLMGraphTransformer (text → entities + relationships → Neo4j)")
print("  Maps to existing: services/memory/semantic_memory.py")

## 8. LangGraph Node Definitions

Every node is a **thin adapter** that delegates to a subagent.  
Pattern: `node(state) → subagent.invoke(context) → partial state update`

Nodes are pure functions that:
1. Extract what they need from `GraphState`
2. Call `subagent.invoke(context)`
3. Return a `dict` with only the keys they modify (LangGraph merges automatically)

In [ ]:
"""
Node definitions — thin adapters wrapping subagents.

10 nodes in the graph:
  input_guard  → GuardrailSubagent (pre-check)
  router       → LLM routing decision
  memory_node  → MemorySubagent.invoke()
  sql_node     → SQLSubagent.invoke()
  fsrs_node    → FSRS scheduling logic
  decision     → multi-source merge + plan
  human_gate   → HITL interrupt() checkpoint
  output_guard → GuardrailSubagent (post-check)
  response     → LLM generation
  post_update  → MemorySubagent.update() + Neo4jSubagent.update()
"""
from langgraph.types import interrupt, Command


# ── Registry (populated at graph build time) ─────────────

_registry: dict[str, Any] = {}  # filled by build_graph()


# ── Node Functions ────────────────────────────────────────

def input_guard_node(state: dict) -> dict:
    """Pre-processing guardrails: PII scan + input validation."""
    guardrail = _registry["guardrail"]
    result = guardrail.invoke({"text": state["user_input"], "direction": "input"})
    if result.get("blocked"):
        return {"generation": result["reason"], "route": "blocked"}
    pii = _registry.get("pii")
    if pii:
        pii_result = pii.invoke({"text": state["user_input"]})
        if pii_result.get("redacted_text"):
            return {"user_input": pii_result["redacted_text"], "pii_detected": pii_result.get("findings", [])}
    return {}


def router_node(state: dict) -> dict:
    """LLM-based routing: decides which workers to activate."""
    llm = _registry["llm"]
    prompt = (
        f"User query: {state['user_input']}\n"
        f"Available workers: memory, sql, fsrs\n"
        "Return a JSON list of worker names to activate. "
        "Return [] if the query can be answered directly."
    )
    response = llm.invoke(prompt)
    # Parse worker list from LLM response
    import json
    try:
        workers = json.loads(response.content)
        if not isinstance(workers, list):
            workers = ["memory"]
    except (json.JSONDecodeError, AttributeError):
        workers = ["memory"]  # default fallback

    return {"active_workers": workers, "plan": f"Routing to: {workers}"}


def memory_node(state: dict) -> dict:
    """Retrieve episodic + semantic memories."""
    memory = _registry["memory"]
    context = {
        "query": state["user_input"],
        "user_id": state["user_id"],
        "already_seen": set(),
        "k": 3,
    }
    result = memory.invoke(context)
    return {
        "retrieved_episodic": result.get("retrieved_episodic", []),
        "retrieved_semantic": result.get("retrieved_semantic", []),
        "memory_context": result.get("memory_context", ""),
    }


def sql_node(state: dict) -> dict:
    """Run SQL retrieval for structured data queries."""
    sql = _registry["sql"]
    result = sql.invoke({"query": state["user_input"], "user_id": state["user_id"]})
    return {"retrieved_sql": result.get("retrieved_sql", "")}


def fsrs_node(state: dict) -> dict:
    """FSRS spaced-repetition scheduling."""
    # Delegates to FSRS service (existing implementation)
    fsrs = _registry.get("fsrs")
    if fsrs:
        result = fsrs.invoke({"query": state["user_input"], "user_id": state["user_id"]})
        return {"fsrs_data": result}
    return {}


def decision_node(state: dict) -> dict:
    """Merge all retrieved context into a unified plan for generation."""
    parts = []
    if state.get("memory_context"):
        parts.append(state["memory_context"])
    if state.get("retrieved_sql"):
        parts.append(f"**Database Results:**\n{state['retrieved_sql']}")
    if state.get("fsrs_data"):
        parts.append(f"**SRS Schedule:**\n{state['fsrs_data']}")

    merged = "\n\n---\n\n".join(parts) if parts else "(no additional context)"
    return {"thought": merged, "iterations": state.get("iterations", 0) + 1}


def human_gate_node(state: dict) -> dict:
    """HITL checkpoint — pauses for human approval if needed.
    Uses LangGraph native interrupt() for pause/resume.
    """
    needs_approval = state.get("needs_human_approval", False)
    if needs_approval:
        # This suspends the graph and waits for Command(resume=...)
        human_response = interrupt({
            "question": "The agent wants to proceed with this plan. Approve?",
            "plan": state.get("plan", ""),
            "context": state.get("thought", ""),
        })
        if human_response.get("approved"):
            return {"human_approved": True}
        else:
            return {"generation": "Action cancelled by user.", "route": "blocked"}
    return {"human_approved": True}


def output_guard_node(state: dict) -> dict:
    """Post-generation guardrails: validate output before returning."""
    guardrail = _registry["guardrail"]
    result = guardrail.invoke({"text": state.get("generation", ""), "direction": "output"})
    if result.get("blocked"):
        return {"generation": "[Response filtered by safety policy]"}
    return {}


def response_node(state: dict) -> dict:
    """Generate final response using LLM + all context."""
    llm = _registry["llm"]
    messages = state.get("messages", [])
    context = state.get("thought", "")

    system_msg = (
        "You are a helpful Japanese language tutor. "
        "Use the following context to inform your response:\n\n"
        f"{context}"
    )
    from langchain_core.messages import SystemMessage, HumanMessage
    full_messages = [SystemMessage(content=system_msg)] + list(messages)
    if not any(isinstance(m, HumanMessage) for m in full_messages):
        full_messages.append(HumanMessage(content=state["user_input"]))

    response = llm.invoke(full_messages)
    return {"generation": response.content, "messages": [response]}


def post_update_node(state: dict) -> dict:
    """Post-response: write to episodic + semantic memory."""
    context = {
        "user_id": state["user_id"],
        "user_input": state["user_input"],
        "generation": state.get("generation", ""),
    }
    results = {}
    memory = _registry.get("memory")
    if memory:
        results.update(memory.update(context))
    neo4j = _registry.get("neo4j")
    if neo4j:
        results.update(neo4j.update(context))
    return results


print("10 node functions defined:")
nodes = [
    "input_guard_node", "router_node", "memory_node", "sql_node",
    "fsrs_node", "decision_node", "human_gate_node", "output_guard_node",
    "response_node", "post_update_node"
]
for n in nodes:
    print(f"  ✓ {n}")

## 9. Human-in-the-Loop (HITL) Lifecycle

LangGraph native HITL using `interrupt()` and `Command(resume=...)`.

**Flow:**
1. Graph reaches `human_gate_node`
2. `interrupt()` suspends execution, saves state to checkpointer
3. Client receives the interrupt payload (question + context)
4. Client sends `Command(resume={"approved": True/False})`
5. Graph resumes from the exact checkpoint

**Checkpointer:** `MemorySaver` (dev) or `AsyncPostgresSaver` (prod) — stores full graph state at every superstep.

In [ ]:
"""
HITL Demo — interrupt/resume lifecycle with MemorySaver checkpointer.

Production: replace MemorySaver with AsyncPostgresSaver backed by Supabase.
"""
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END


# Minimal HITL demo graph
def build_hitl_demo():
    """Demonstrates the interrupt → resume lifecycle."""

    def ask_node(state: dict) -> dict:
        """Node that pauses for human approval."""
        answer = interrupt({"question": "Shall I proceed?", "draft": state.get("draft", "")})
        return {"approved": answer.get("approved", False)}

    def act_node(state: dict) -> dict:
        if state.get("approved"):
            return {"result": "Action completed!"}
        return {"result": "Action cancelled by user."}

    builder = StateGraph(dict)
    builder.add_node("ask", ask_node)
    builder.add_node("act", act_node)
    builder.add_edge(START, "ask")
    builder.add_edge("ask", "act")
    builder.add_edge("act", END)

    checkpointer = MemorySaver()
    return builder.compile(checkpointer=checkpointer)


# Simulate lifecycle
graph = build_hitl_demo()
thread_config = {"configurable": {"thread_id": "hitl-demo-1"}}

# Step 1: Start — graph will pause at interrupt()
print("Step 1: Starting graph (will pause at interrupt)...")
for event in graph.stream({"draft": "Review this plan"}, config=thread_config):
    print(f"  Event: {event}")

# Step 2: Check state
state = graph.get_state(thread_config)
print(f"\nStep 2: Graph paused. Next node: {state.next}")
print(f"  Interrupt value: {state.tasks}")

# Step 3: Resume with approval
print("\nStep 3: Resuming with approval...")
for event in graph.stream(Command(resume={"approved": True}), config=thread_config):
    print(f"  Event: {event}")

print("\n✅ HITL lifecycle complete: interrupt → checkpoint → resume")

## 10. Graph Assembly — `build_graph()`

Full graph construction with all nodes, edges, and conditional routing.

**Conditional edge functions:**
- `route_after_input_guard` → blocked or router
- `route_workers` → fan-out to active workers
- `route_after_decision` → human_gate or response (based on needs_human_approval)
- `route_after_human_gate` → response or END (if blocked)
- `should_loop` → decision (if iterations < max) or response (terminate)

In [ ]:
"""
Full graph assembly — the central build_graph() function.

This replaces the existing _build_graph() in graph.py with:
- All 10 nodes
- Conditional edges for routing, HITL, and loop termination
- Configurable constraints (MAX_ITERATIONS, GLOBAL_TIMEOUT)
"""
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import Literal

# ── Constraints ───────────────────────────────────────────

CONSTRAINTS = {
    "MAX_ITERATIONS": 5,
    "GLOBAL_TIMEOUT_S": 50.0,
    "MAX_WORKER_TIMEOUT_S": 10.0,
}


# ── Conditional Edge Functions ────────────────────────────

def route_after_input_guard(state: dict) -> Literal["router", "__end__"]:
    """If input was blocked by guardrails, skip to END."""
    if state.get("route") == "blocked":
        return "__end__"
    return "router"


def route_workers(state: dict) -> list[str]:
    """Fan-out: activate the workers chosen by the router."""
    workers = state.get("active_workers", [])
    valid = {"memory_node", "sql_node", "fsrs_node"}
    # Map short names to node names
    name_map = {"memory": "memory_node", "sql": "sql_node", "fsrs": "fsrs_node"}
    activated = [name_map[w] for w in workers if w in name_map]
    return activated if activated else ["memory_node"]  # default to memory


def route_after_decision(state: dict) -> Literal["human_gate", "response"]:
    """If human approval needed, go to HITL gate; otherwise generate."""
    if state.get("needs_human_approval"):
        return "human_gate"
    return "response"


def route_after_human_gate(state: dict) -> Literal["response", "__end__"]:
    """After HITL: proceed to response or abort."""
    if state.get("route") == "blocked":
        return "__end__"
    return "response"


def should_loop(state: dict) -> Literal["decision", "response"]:
    """Check iteration count for loop termination."""
    iterations = state.get("iterations", 0)
    max_iter = CONSTRAINTS["MAX_ITERATIONS"]
    if iterations >= max_iter:
        return "response"  # force exit
    return "decision"


# ── Graph Builder ─────────────────────────────────────────

def build_graph(
    llm=None,
    memory_subagent=None,
    sql_subagent=None,
    neo4j_subagent=None,
    guardrail_subagent=None,
    pii_subagent=None,
    fsrs_subagent=None,
    checkpointer=None,
):
    """Construct the full LangGraph state machine.

    Args:
        llm: ChatOpenAI or compatible LLM
        memory_subagent: MemorySubagent instance
        sql_subagent: SQLSubagent instance
        neo4j_subagent: Neo4jSubagent instance
        guardrail_subagent: GuardrailSubagent instance
        pii_subagent: PIISubagent (optional)
        fsrs_subagent: FSRS scheduling subagent (optional)
        checkpointer: LangGraph checkpointer (MemorySaver or AsyncPostgresSaver)
    """
    # Register subagents in the module-level registry
    global _registry
    _registry = {
        "llm": llm,
        "memory": memory_subagent,
        "sql": sql_subagent,
        "neo4j": neo4j_subagent,
        "guardrail": guardrail_subagent,
        "pii": pii_subagent,
        "fsrs": fsrs_subagent,
    }

    if checkpointer is None:
        checkpointer = MemorySaver()

    builder = StateGraph(dict)  # Production: use GraphState TypedDict

    # ── Add all nodes ───────────────────────────────────
    builder.add_node("input_guard", input_guard_node)
    builder.add_node("router", router_node)
    builder.add_node("memory_node", memory_node)
    builder.add_node("sql_node", sql_node)
    builder.add_node("fsrs_node", fsrs_node)
    builder.add_node("decision", decision_node)
    builder.add_node("human_gate", human_gate_node)
    builder.add_node("response", response_node)
    builder.add_node("output_guard", output_guard_node)
    builder.add_node("post_update", post_update_node)

    # ── Edges ───────────────────────────────────────────

    # Entry: START → input_guard
    builder.add_edge(START, "input_guard")

    # input_guard → router (or END if blocked)
    builder.add_conditional_edges("input_guard", route_after_input_guard)

    # router → fan-out to workers
    builder.add_conditional_edges("router", route_workers)

    # All workers → decision (fan-in)
    builder.add_edge("memory_node", "decision")
    builder.add_edge("sql_node", "decision")
    builder.add_edge("fsrs_node", "decision")

    # decision → human_gate or response
    builder.add_conditional_edges("decision", route_after_decision)

    # human_gate → response or END
    builder.add_conditional_edges("human_gate", route_after_human_gate)

    # response → output_guard
    builder.add_edge("response", "output_guard")

    # output_guard → post_update
    builder.add_edge("output_guard", "post_update")

    # post_update → END
    builder.add_edge("post_update", END)

    return builder.compile(checkpointer=checkpointer)


print("build_graph() defined with:")
print(f"  Nodes: 10")
print(f"  Conditional edges: 5 (input_guard, router, decision, human_gate, loop)")
print(f"  Constraints: {CONSTRAINTS}")
print(f"  Checkpointer: configurable (MemorySaver default)")
print()
print("Graph flow:")
print("  START → input_guard → router → [memory|sql|fsrs] → decision")
print("       → human_gate? → response → output_guard → post_update → END")

## 11. Observability — Execution Trace Handler

LangChain callback handler that captures per-node timing, token usage, and error traces.  
Integrates with existing `tracing.py` and emits structured logs for every graph execution.

In [ ]:
"""
Observability — ExecutionTraceHandler callback.

Captures per-node timing, LLM token usage, and error events.
Designed to plug into LangGraph's callback system.
"""
import time
import logging
from dataclasses import dataclass, field
from langchain_core.callbacks import BaseCallbackHandler

logger = logging.getLogger("hanchan.trace")


@dataclass
class NodeTrace:
    """Trace for a single node execution."""
    node_name: str
    start_time: float = 0.0
    end_time: float = 0.0
    duration_ms: float = 0.0
    tokens_in: int = 0
    tokens_out: int = 0
    error: str | None = None


@dataclass
class ExecutionTrace:
    """Full trace for one graph invocation."""
    thread_id: str = ""
    user_id: str = ""
    start_time: float = 0.0
    total_duration_ms: float = 0.0
    nodes: list[NodeTrace] = field(default_factory=list)
    total_tokens_in: int = 0
    total_tokens_out: int = 0

    def summary(self) -> dict:
        return {
            "thread_id": self.thread_id,
            "user_id": self.user_id,
            "total_ms": round(self.total_duration_ms, 1),
            "node_count": len(self.nodes),
            "total_tokens": self.total_tokens_in + self.total_tokens_out,
            "nodes": [
                {"name": n.node_name, "ms": round(n.duration_ms, 1), "error": n.error}
                for n in self.nodes
            ],
        }


class ExecutionTraceHandler(BaseCallbackHandler):
    """LangChain callback handler for graph execution tracing."""

    def __init__(self):
        self._trace = ExecutionTrace()
        self._current_node: NodeTrace | None = None

    @property
    def trace(self) -> ExecutionTrace:
        return self._trace

    def on_chain_start(self, serialized, inputs, **kwargs):
        if not self._trace.start_time:
            self._trace.start_time = time.monotonic()
            self._trace.thread_id = kwargs.get("config", {}).get("configurable", {}).get("thread_id", "")

    def on_chain_end(self, outputs, **kwargs):
        if self._current_node:
            self._current_node.end_time = time.monotonic()
            self._current_node.duration_ms = (self._current_node.end_time - self._current_node.start_time) * 1000
            self._trace.nodes.append(self._current_node)
            self._current_node = None

        self._trace.total_duration_ms = (time.monotonic() - self._trace.start_time) * 1000

    def on_llm_start(self, serialized, prompts, **kwargs):
        node_name = kwargs.get("tags", ["unknown"])[0] if kwargs.get("tags") else "llm"
        self._current_node = NodeTrace(node_name=node_name, start_time=time.monotonic())

    def on_llm_end(self, response, **kwargs):
        if self._current_node and response.llm_output:
            usage = response.llm_output.get("token_usage", {})
            self._current_node.tokens_in = usage.get("prompt_tokens", 0)
            self._current_node.tokens_out = usage.get("completion_tokens", 0)
            self._trace.total_tokens_in += self._current_node.tokens_in
            self._trace.total_tokens_out += self._current_node.tokens_out

    def on_chain_error(self, error, **kwargs):
        if self._current_node:
            self._current_node.error = str(error)
        logger.error(f"Chain error: {error}")


# Demo
handler = ExecutionTraceHandler()
print("ExecutionTraceHandler defined:")
print("  Captures: per-node timing, token counts, errors")
print("  Usage: graph.invoke(input, config={'callbacks': [handler]})")
print("  Access: handler.trace.summary()")
print()
print("  Maps to existing: core/tracing.py")

## 12. Constraints & Loop Termination

Hard limits that prevent runaway execution:

| Constraint | Value | Enforcement |
|---|---|---|
| `MAX_ITERATIONS` | 5 | `should_loop()` conditional edge forces exit to response |
| `GLOBAL_TIMEOUT_S` | 50.0 | Checked in decision_node; if exceeded, skip to response |
| `MAX_WORKER_TIMEOUT_S` | 10.0 | Per-worker asyncio.wait_for() wrapper |
| Blocked route | - | `route_after_input_guard` / `route_after_human_gate` → END |

Each constraint maps to a conditional edge function, keeping the logic declarative and testable.

## 13. Architecture Diagrams

### Graph Topology (Mermaid)

```mermaid
stateDiagram-v2
    [*] --> input_guard
    input_guard --> router : pass
    input_guard --> [*] : blocked

    router --> memory_node : needs_memory
    router --> sql_node : needs_sql
    router --> fsrs_node : needs_fsrs

    memory_node --> decision
    sql_node --> decision
    fsrs_node --> decision

    decision --> human_gate : needs_approval
    decision --> response : auto_approve

    human_gate --> response : approved
    human_gate --> [*] : rejected

    response --> output_guard
    output_guard --> post_update
    post_update --> [*]
```

### Layered Architecture

```mermaid
graph TB
    subgraph "API Layer"
        FastAPI[FastAPI Router]
    end

    subgraph "Graph Layer"
        SG[StateGraph]
        CP[Checkpointer]
    end

    subgraph "Node Layer"
        IG[input_guard]
        RT[router]
        MN[memory_node]
        SN[sql_node]
        FN[fsrs_node]
        DN[decision]
        HG[human_gate]
        RN[response]
        OG[output_guard]
        PU[post_update]
    end

    subgraph "Subagent Layer"
        GS[GuardrailSubagent]
        MS[MemorySubagent]
        SS[SQLSubagent]
        NS[Neo4jSubagent]
        PS[PIISubagent]
    end

    subgraph "Integration Layer"
        Qdrant[(Qdrant)]
        Neo4j[(Neo4j)]
        Supabase[(Supabase/PG)]
        LLM[LLM Proxy]
    end

    FastAPI --> SG
    SG --> IG & RT & MN & SN & FN & DN & HG & RN & OG & PU
    IG --> GS & PS
    MN --> MS
    SN --> SS
    PU --> MS & NS
    MS --> Qdrant
    NS --> Neo4j
    SS --> Supabase
    RN --> LLM
```

## 14. Module Structure & Integration Mapping

### Target directory layout

```
src/fastapi/app/agents/memory_agent/
├── graph.py                    # build_graph() — graph assembly
├── state.py                    # GraphState TypedDict
├── constraints.py              # CONSTRAINTS dict + timeout helpers
├── nodes/
│   ├── __init__.py
│   ├── input_guard.py          # input_guard_node
│   ├── router.py               # router_node
│   ├── workers.py              # memory_node, sql_node, fsrs_node
│   ├── decision.py             # decision_node
│   ├── human_gate.py           # human_gate_node
│   ├── response.py             # response_node
│   ├── output_guard.py         # output_guard_node
│   └── post_update.py          # post_update_node
└── subagents/
    ├── __init__.py
    ├── base.py                 # Subagent Protocol + SubagentRegistry
    ├── guardrail.py            # GuardrailSubagent + PIISubagent
    ├── memory.py               # MemorySubagent
    ├── sql.py                  # SQLSubagent
    └── neo4j.py                # Neo4jSubagent

src/fastapi/app/core/
├── config.py                   # Settings (existing, add new env vars)
├── llm.py                      # make_llm(), make_embedding_model() (existing)
├── tracing.py                  # ExecutionTraceHandler (refactored)
└── observability.py            # ExecutionTrace, NodeTrace dataclasses

src/fastapi/app/services/memory/
├── episodic_memory.py          # Implements VectorStoreProtocol (existing)
├── semantic_memory.py          # Implements GraphStoreProtocol (existing)
├── session_memory.py           # Session context (existing)
└── consolidation.py            # Periodic memory consolidation (existing)
```

### Subagent → LangChain Integration Mapping

| Subagent | LangChain Component | External Service |
|---|---|---|
| `GuardrailSubagent` | `ChatPromptTemplate` (LLM judge) | LLM Proxy |
| `PIISubagent` | Regex + optional NER | None (local) |
| `MemorySubagent` | `QdrantVectorStore`, `Neo4jGraph` | Qdrant Cloud, Neo4j Aura |
| `SQLSubagent` | `SQLDatabaseToolkit`, `AgentExecutor` | Supabase (PostgreSQL) |
| `Neo4jSubagent` | `GraphCypherQAChain`, `LLMGraphTransformer` | Neo4j Aura |

### Node → Subagent Mapping

| Node | Subagent(s) | State Keys Written |
|---|---|---|
| `input_guard` | GuardrailSubagent, PIISubagent | `route`, `pii_detected` |
| `router` | LLM (direct) | `active_workers`, `plan` |
| `memory_node` | MemorySubagent | `retrieved_episodic`, `retrieved_semantic`, `memory_context` |
| `sql_node` | SQLSubagent | `retrieved_sql` |
| `fsrs_node` | FSRS service | `fsrs_data` |
| `decision` | None (pure logic) | `thought`, `iterations` |
| `human_gate` | None (interrupt) | `human_approved` |
| `response` | LLM (direct) | `generation`, `messages` |
| `output_guard` | GuardrailSubagent | `generation` (filtered) |
| `post_update` | MemorySubagent, Neo4jSubagent | episodic/semantic write results |

### Integration → Module Mapping

| Integration | Existing Module | New Location |
|---|---|---|
| Qdrant | `services/memory/episodic_memory.py` | Wrapped by `subagents/memory.py` |
| Neo4j | `services/memory/semantic_memory.py` | Wrapped by `subagents/neo4j.py` |
| Supabase SQL | `services/sql/` | Wrapped by `subagents/sql.py` |
| LLM Proxy | `core/llm.py` | Unchanged (used by nodes directly) |
| FSRS | `services/fsrs/` | Wrapped by dedicated node |
| Checkpointer | New | `graph.py` (MemorySaver / AsyncPostgresSaver) |

## 15. Summary — Before vs After

| Aspect | Before (Current) | After (This Design) |
|---|---|---|
| **Graph structure** | Flat orchestrator → workers → reviewer → generate | Layered: guard → route → workers → decision → HITL → response → guard → update |
| **Subagent pattern** | Workers are node functions with inline logic | Subagent Protocol with registry, testable in isolation |
| **Guardrails** | None | 3-layer: deterministic → PII → LLM judge, both input and output |
| **PII handling** | None | PIISubagent with regex + configurable strategies (redact/mask/hash/block) |
| **Memory retrieval** | Separate episodic/semantic calls in workers | Unified MemorySubagent with dedup and merged context |
| **Memory update** | Best-effort in workers | Dedicated post_update node, always runs after response |
| **SQL access** | Direct DB queries in sql_worker | SQLDatabaseToolkit with retry + dedup + read-only enforcement |
| **Neo4j** | Direct Cypher in semantic_memory | GraphCypherQAChain (read) + LLMGraphTransformer (write) |
| **HITL** | None | LangGraph native interrupt() + Command(resume=) with checkpointer |
| **State** | 20-field TypedDict | 26-field TypedDict with Annotated reducers for lists/sets |
| **Loop control** | MAX_ITERATIONS + timeout in orchestrator | CONSTRAINTS dict + conditional edge functions (declarative) |
| **Observability** | Basic logging | ExecutionTraceHandler callback with per-node timing + token tracking |
| **Checkpointing** | None | MemorySaver (dev) / AsyncPostgresSaver (prod) |
| **Testability** | Requires full graph | Subagents testable in isolation via Protocol interface |

### Key Design Decisions

1. **Subagent Protocol pattern** — every capability is a class with `.invoke(context) → dict`. Nodes are thin adapters. This makes unit testing trivial (mock the subagent, test the node).

2. **Fan-out/fan-in routing** — router decides which workers activate; LangGraph handles parallel execution. Workers merge at decision node.

3. **Guardrails sandwich** — input_guard before any processing, output_guard after generation. Both delegate to the same GuardrailSubagent with direction awareness.

4. **Post-update as dedicated node** — memory writes happen after the response is generated, never during retrieval. Prevents partial writes on error.

5. **HITL via native interrupt()** — no custom polling or webhook machinery. LangGraph handles state persistence and resumption natively via checkpointer.

6. **Existing services preserved** — `episodic_memory.py`, `semantic_memory.py`, `core/llm.py`, `core/config.py` are wrapped, not replaced. Migration is incremental.

---

*Architecture design complete. Next step: implement subagents and nodes following this blueprint.*